# Component_01 — STAGE 5: Classifier Retraining
### ConvNeXt-Base · 8 pathologies · fused labels · per-image normalisation

This is **Model 1** — the network that predicts cardiomegaly and the
co-pathologies from a chest X-ray, and supplies Grad-CAM evidence.

It is trained **before** Model 2 because the report generator loads this
model's frozen backbone as its vision encoder.

---

## What changed since your last training run

| | Before | Now | Source |
|---|---|---|---|
| Labels | custom NLP labeller | **text-adjudicated fusion** (wins 8/8 on precision) | Stage 3 |
| Normalisation | ImageNet on grayscale | **per-image z-score** | Stage 2 |
| Augmentation | ColorJitter + Autocontrast + 10° | **geometric only, 5°** | Stage 2 |
| Class imbalance | none — `LABEL_IMPORTANCE` only scales per-label loss | **`pos_weight` clamped at 8** | this stage |
| Uncertain labels | ignored | **downweighted 0.5×** | this stage |
| Model selection | Cardiomegaly AUROC only | **mean AUROC over 8** | this stage |
| Thresholds | fixed 0.5 | **per-class F1-optimal, fitted on val** | this stage |
| Weight averaging | none | **EMA** | this stage |
| Image paths | inferred from label (131 silently lost) | **resolved + verified** | Stage 3.5 |

---

## 💰 Compute-unit budget — read before starting

**Colab Pro = $9.99 = 100 compute units.**

| GPU | CU/hr | Hours from 100 CU | Use |
|---|---|---|---|
| CPU | **0** | ∞ | Stages 1–3 |
| T4 | ~1.8 | ~57 | too slow, no bf16 |
| **L4** | **~4.8** | **~21** | ⭐ **this stage** |
| A100 | ~15 | ~7 | burns too fast |

**Plan: Stage 5 ≈ 4–6 h on L4 ≈ 20–29 CU.** Leaves ~70 CU for Stage 4.

### The rule that saves you a re-run

`SMOKE_TEST = True` runs 40 steps + a full eval pass on a subset in ~5 minutes
(≈0.4 CU). It exercises **every** code path — data, model, loss, EMA, eval,
checkpoint save, checkpoint resume. Only after it prints `SMOKE TEST PASSED` do
you set it to `False` and commit to the real run.

⚠️ **Do not skip the smoke test.** Five minutes now versus five hours wasted.

---
# 0 · Configuration

Everything you might change lives in this one cell.

In [ ]:
CFG = dict(
    # ---- run control -------------------------------------------------------
    SMOKE_TEST      = True,     # ⚠️ leave True for the first run. Then set False.
    RESUME          = True,     # auto-resume from the last Drive checkpoint

    # ---- data --------------------------------------------------------------
    IMG_SIZE        = 384,
    NUM_WORKERS     = 4,        # Colab gives 2 vCPU on most tiers; 4 is safe
    PIN_MEMORY      = True,

    # ---- model -------------------------------------------------------------
    ARCH            = "convnext_base",
    PRETRAINED      = True,     # ImageNet. NOT the old checkpoint — see note below
    DROPOUT         = 0.3,

    # ---- optimisation ------------------------------------------------------
    EPOCHS          = 30,
    EFFECTIVE_BATCH = 64,       # reached via grad accumulation
    BASE_LR         = 2e-4,
    WEIGHT_DECAY    = 0.05,
    WARMUP_EPOCHS   = 3,
    LABEL_SMOOTH    = 0.0,
    GRAD_CLIP       = 1.0,
    PATIENCE        = 8,

    # ---- loss --------------------------------------------------------------
    POS_WEIGHT_CLAMP = 8.0,     # raw values reach 24 → unstable. 8 is standard.
    UNCERTAIN_WEIGHT = 0.5,     # masking would delete 40% of Pneumonia positives

    # ---- regularisation ----------------------------------------------------
    USE_EMA         = True,
    EMA_DECAY       = 0.9998,

    # ---- speed -------------------------------------------------------------
    AMP_DTYPE       = "bf16",   # L4/A100 yes. T4 → set "fp16".
    CHANNELS_LAST   = True,
    USE_COMPILE     = False,    # ~20% faster but occasionally fails; off for safety

    SEED            = 42,
)
PATHOLOGIES = ["Cardiomegaly", "Edema", "Pleural_Effusion", "Atelectasis",
               "Consolidation", "Lung_Opacity", "Pneumonia", "Pneumothorax"]
print("SMOKE_TEST =", CFG["SMOKE_TEST"], "  <-- must be False for the real run")

> **Why start from ImageNet and not your existing checkpoint?**
> `models/cardio_classifier/best_model.pt` was trained under ImageNet
> normalisation. Stage 2 replaced that with per-image z-score, so the input
> distribution its early layers expect no longer exists. Starting fresh costs a
> few epochs and removes a whole class of silent degradation.

---
# 1 · Environment & GPU check

In [ ]:
import os, sys, json, math, time, random, subprocess, importlib, warnings, gc
warnings.filterwarnings("ignore")
from pathlib import Path
from datetime import datetime

import numpy as np

print("=" * 80)
print("  COMPONENT_01 · STAGE 5 · CLASSIFIER RETRAINING")
print("=" * 80)

try:
    smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                          "--format=csv,noheader"], capture_output=True, text=True, timeout=15)
    GPU = smi.stdout.strip()
except Exception:
    GPU = ""

if not GPU:
    print("  ❌ NO GPU DETECTED")
    print("     Runtime → Change runtime type → GPU → L4, then re-run.")
    raise SystemExit("GPU required for Stage 5")

print(f"  GPU: {GPU}")
GPU_NAME = GPU.split(",")[0].strip()
CU_PER_HR = {"L4": 4.8, "A100": 15.0, "T4": 1.76, "V100": 4.9}
RATE = next((v for k, v in CU_PER_HR.items() if k in GPU_NAME), 5.0)
print(f"  estimated burn rate: ~{RATE} compute units/hour")
if "T4" in GPU_NAME:
    print("  ⚠️  T4 has no bf16 — setting AMP_DTYPE='fp16'")
    CFG["AMP_DTYPE"] = "fp16"
if "A100" in GPU_NAME:
    print("  ⚠️  A100 burns ~15 CU/hr → only ~7 h from 100 units. L4 is the better buy.")

import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import pandas as pd
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.benchmark = True
seed_all(CFG["SEED"])

# Colab vCPU count varies by tier; more workers than cores just thrashes (audit fix E)
CFG["NUM_WORKERS"] = max(2, min(CFG["NUM_WORKERS"], (os.cpu_count() or 4)))
print(f"  vCPUs={os.cpu_count()} -> NUM_WORKERS={CFG['NUM_WORKERS']}")
DEV = torch.device("cuda")
print(f"  torch {torch.__version__} | VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print("=" * 80)
T_START = time.time()

---
# 2 · Mount Drive, extract images, verify

Extracting the tar to `/content/` is what makes training fast — every image
read then hits Colab's local SSD instead of Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

PROJECT   = Path("/content/drive/MyDrive/Component_01")
TAR       = PROJECT / "data" / "images" / "cardio_384.tar"
MANIFEST  = PROJECT / "training_manifest"
CKPT_DIR  = PROJECT / "checkpoints" / "stage5"
REPORTS   = PROJECT / "reports" / "stage5"
IMG_ROOT  = Path("/content/cardio_image_384")
for d in (CKPT_DIR, REPORTS):
    d.mkdir(parents=True, exist_ok=True)

if not IMG_ROOT.exists():
    if not TAR.exists():
        raise FileNotFoundError(
            f"{TAR} not found.\nUpload cardio_384.tar to MyDrive/Component_01/data/images/")
    print(f"extracting {TAR.stat().st_size/1e9:.1f} GB tar → /content/ ...")
    t0 = time.time()
    subprocess.run(["tar", "-xf", str(TAR), "-C", "/content/"], check=True)
    print(f"  done in {time.time()-t0:.0f}s")
else:
    print("images already extracted this session")

n_png = sum(1 for _ in IMG_ROOT.rglob("*.png"))
print(f"  images on local disk: {n_png:,}")
assert n_png >= 45000, f"expected ~46,274 PNGs, found {n_png:,} — re-extract the tar"

---
# 3 · Load manifests

In [ ]:
for s in ("train", "val", "test"):
    p = MANIFEST / f"manifest_{s}.csv"
    if not p.exists():
        raise FileNotFoundError(
            f"{p} not found.\nUpload Component_01/training_manifest/ to "
            f"MyDrive/Component_01/training_manifest/")

DF = {s: pd.read_csv(MANIFEST / f"manifest_{s}.csv", low_memory=False)
      for s in ("train", "val", "test")}
for s, d in DF.items():
    print(f"  {s:<6}{len(d):>7,} rows")
    assert d["dicom_id"].is_unique
    for p in PATHOLOGIES:
        assert p in d.columns, f"{p} missing"
        assert set(d[p].unique()) <= {0, 1}, f"{p} not binary"
        assert f"{p}_uncertain" in d.columns

sub = {s: set(d.subject_id) for s, d in DF.items()}
leak = len(sub["train"] & sub["val"]) + len(sub["train"] & sub["test"]) + len(sub["val"] & sub["test"])
assert leak == 0, f"PATIENT LEAKAGE: {leak}"
print(f"  ✅ zero patient leakage | mean labels/image {(DF['train'][PATHOLOGIES]==1).sum(axis=1).mean():.2f}")

missing = sum(1 for p in DF["train"]["image_path"].head(500) if not (IMG_ROOT / p).exists())
assert missing == 0, f"{missing}/500 sampled train images not found"
print("  ✅ image paths resolve")

print(f"\n  {'pathology':<22}{'train pos':>10}{'prev%':>8}{'pos_weight':>12}")
POS_W = []
for p in PATHOLOGIES:
    pos = int((DF["train"][p] == 1).sum()); neg = len(DF["train"]) - pos
    w = min(neg / max(pos, 1), CFG["POS_WEIGHT_CLAMP"])
    POS_W.append(w)
    print(f"  {p:<22}{pos:>10,}{pos/len(DF['train'])*100:>7.2f}%{w:>12.2f}")
POS_W = torch.tensor(POS_W, dtype=torch.float32, device=DEV)

---
# 4 · Transforms (Stage 2 pipeline, written inline so the notebook is self-contained)

In [ ]:
_EPS = 1e-6

class ToGrayscalePIL:
    def __call__(self, img):
        return img if img.mode == "L" else img.convert("L")

class PerImageZScore:
    """(1,H,W) in [0,1] -> (3,H,W), per-image mean 0 / std 1.
    The std guard prevents a constant frame producing NaN and poisoning the run."""
    def __init__(self, c=3): self.c = c
    def __call__(self, t):
        if t.shape[0] != 1: t = t[:1]
        s = t.std()
        t = (t - t.mean()) / s if s > _EPS else t - t.mean()
        return t.repeat(self.c, 1, 1)

def build_transform(split, img_size=CFG["IMG_SIZE"]):
    ops = [ToGrayscalePIL(), transforms.Resize((img_size, img_size))]
    if split == "train":
        # geometric only — no flip (laterality is diagnostic),
        # no photometric jitter (intensity IS the signal for edema/opacity)
        ops.append(transforms.RandomAffine(
            degrees=5.0, translate=(0.03, 0.03), scale=(0.97, 1.03),
            interpolation=transforms.InterpolationMode.BILINEAR, fill=0))
    ops += [transforms.ToTensor(), PerImageZScore(3)]
    return transforms.Compose(ops)

_t = build_transform("eval")(Image.open(IMG_ROOT / DF["train"]["image_path"].iloc[0]))
print(f"  sample tensor {tuple(_t.shape)} mean={_t.mean():.2e} std={_t.std():.4f}")
assert tuple(_t.shape) == (3, CFG["IMG_SIZE"], CFG["IMG_SIZE"]) and torch.isfinite(_t).all()
print("  ✅ transform verified")

---
# 5 · Dataset

In [ ]:
class CXRDataset(Dataset):
    def __init__(self, df, split, root=IMG_ROOT):
        self.paths = [str(root / p) for p in df["image_path"]]
        self.y = df[PATHOLOGIES].values.astype(np.float32)
        self.u = df[[f"{p}_uncertain" for p in PATHOLOGIES]].values.astype(np.float32)
        self.tf = build_transform(split)
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = self.tf(Image.open(self.paths[i]))
        # weight 1.0 for confident cells, UNCERTAIN_WEIGHT for uncertain ones
        w = 1.0 - self.u[i] * (1.0 - CFG["UNCERTAIN_WEIGHT"])
        return img, torch.from_numpy(self.y[i]), torch.from_numpy(w)

DS = {s: CXRDataset(DF[s], "train" if s == "train" else "eval") for s in DF}
print("  " + " | ".join(f"{s}={len(d):,}" for s, d in DS.items()))

---
# 6 · Model, loss, EMA

In [ ]:
class CXRClassifier(nn.Module):
    """ConvNeXt-Base + multi-label head. `self.features` is the Grad-CAM hook
    point and is what Stage 4 loads as its frozen vision encoder."""
    def __init__(self, n=len(PATHOLOGIES), pretrained=True, p_drop=CFG["DROPOUT"]):
        super().__init__()
        w = models.ConvNeXt_Base_Weights.IMAGENET1K_V1 if pretrained else None
        base = models.convnext_base(weights=w)
        self.features = base.features
        self.avgpool = base.avgpool
        d = base.classifier[2].in_features           # 1024
        self.classifier = nn.Sequential(
            nn.LayerNorm(d), nn.Dropout(p_drop), nn.Linear(d, 512),
            nn.GELU(), nn.Dropout(p_drop * 0.66), nn.Linear(512, n))
    def forward(self, x):
        return self.classifier(self.avgpool(self.features(x)).flatten(1))

class WeightedBCE(nn.Module):
    """BCEWithLogits with per-class pos_weight AND per-cell confidence weights.

    pos_weight fixes pos/neg imbalance (the previous run had none — at 4%
    prevalence the model is rewarded for always predicting 'no').
    The per-cell weight downweights labels flagged uncertain in Stage 3."""
    def __init__(self, pos_weight):
        super().__init__()
        self.register_buffer("pw", pos_weight)
    def forward(self, logits, targets, cellw):
        l = nn.functional.binary_cross_entropy_with_logits(
            logits, targets, pos_weight=self.pw, reduction="none")
        return (l * cellw).sum() / cellw.sum().clamp(min=1.0)

class EMA:
    """Exponential moving average of weights.

    BUG FIX (audit B): the first version evaluated by doing
    `copy.deepcopy(model)` each epoch. That allocates a SECOND 89M-parameter
    model on the GPU — immediately after the batch-size finder has already
    pushed VRAM to its limit — so epoch 1's evaluation would OOM and destroy
    the run. Instead we swap the EMA weights into the live model, evaluate, and
    swap the originals back. Peak memory cost is one CPU-side copy, not a GPU one."""
    def __init__(self, model, decay):
        self.decay = decay
        self.shadow = {k: v.detach().clone().float()
                       for k, v in model.state_dict().items() if v.dtype.is_floating_point}
        self._backup = None
    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if k in self.shadow:
                self.shadow[k].mul_(self.decay).add_(v.detach().float(), alpha=1 - self.decay)
    def copy_to(self, model):
        sd = model.state_dict()
        for k, v in self.shadow.items():
            sd[k].copy_(v)
    @torch.no_grad()
    def apply(self, model):
        """Swap EMA weights in, keeping the originals on CPU."""
        sd = model.state_dict()
        self._backup = {k: sd[k].detach().to("cpu", copy=True) for k in self.shadow}
        for k, v in self.shadow.items():
            sd[k].copy_(v)
    @torch.no_grad()
    def restore(self, model):
        """Put the training weights back."""
        if self._backup is None:
            return
        sd = model.state_dict()
        for k, v in self._backup.items():
            sd[k].copy_(v.to(sd[k].device))
        self._backup = None

model = CXRClassifier(pretrained=CFG["PRETRAINED"]).to(DEV)
if CFG["CHANNELS_LAST"]:
    model = model.to(memory_format=torch.channels_last)
criterion = WeightedBCE(POS_W).to(DEV)
print(f"  params {sum(p.numel() for p in model.parameters())/1e6:.1f}M | "
      f"channels_last={CFG['CHANNELS_LAST']} | AMP={CFG['AMP_DTYPE']}")

---
# 7 · Automatic batch-size finder

OOM three hours into a run is the classic way to waste compute units. This
probes real forward+backward passes and picks the largest size that fits.

In [ ]:
AMP_DT = torch.bfloat16 if CFG["AMP_DTYPE"] == "bf16" else torch.float16

def probe(bs):
    try:
        torch.cuda.empty_cache(); gc.collect()
        x = torch.randn(bs, 3, CFG["IMG_SIZE"], CFG["IMG_SIZE"], device=DEV)
        if CFG["CHANNELS_LAST"]: x = x.to(memory_format=torch.channels_last)
        y = torch.randint(0, 2, (bs, len(PATHOLOGIES)), device=DEV).float()
        w = torch.ones_like(y)
        # lr=0.0 — the probe must NOT nudge the pretrained weights (audit fix C)
        opt = torch.optim.AdamW(model.parameters(), lr=0.0)
        with torch.autocast("cuda", dtype=AMP_DT):
            loss = criterion(model(x), y, w)
        loss.backward(); opt.step(); opt.zero_grad(set_to_none=True)
        peak = torch.cuda.max_memory_allocated() / 1e9
        del x, y, w, opt, loss
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); gc.collect()
        return True, peak
    except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
        # older torch raises a plain RuntimeError for OOM (audit fix D)
        if 'out of memory' not in str(e).lower() and not isinstance(e, torch.cuda.OutOfMemoryError):
            raise
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); gc.collect()
        return False, 0.0

BATCH = None
for bs in (48, 32, 24, 16, 12, 8):
    ok, peak = probe(bs)
    print(f"  batch {bs:>3}: {'OK  ' if ok else 'OOM '} {f'peak {peak:.1f} GB' if ok else ''}")
    if ok:
        BATCH = bs; break
assert BATCH, "even batch 8 does not fit — use a smaller IMG_SIZE"
ACCUM = max(1, round(CFG["EFFECTIVE_BATCH"] / BATCH))
print(f"\n  batch={BATCH} × accum={ACCUM} → effective {BATCH*ACCUM}")
model.zero_grad(set_to_none=True)

---
# 8 · Dataloaders, optimiser, schedule

In [ ]:
LOADERS = {
    "train": DataLoader(DS["train"], batch_size=BATCH, shuffle=True,
                        num_workers=CFG["NUM_WORKERS"], pin_memory=CFG["PIN_MEMORY"],
                        drop_last=True, persistent_workers=CFG["NUM_WORKERS"] > 0),
    # eval batch == train batch: 2x had no memory headroom after the
    # batch finder had already maximised VRAM (audit fix H)
    "val":   DataLoader(DS["val"], batch_size=BATCH, shuffle=False,
                        num_workers=CFG["NUM_WORKERS"], pin_memory=CFG["PIN_MEMORY"]),
    "test":  DataLoader(DS["test"], batch_size=BATCH, shuffle=False,
                        num_workers=CFG["NUM_WORKERS"], pin_memory=CFG["PIN_MEMORY"]),
}
# LR scaled linearly with effective batch (Goyal et al. linear scaling rule)
LR = CFG["BASE_LR"] * (BATCH * ACCUM) / 64
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=CFG["WEIGHT_DECAY"])
steps_per_epoch = max(1, len(LOADERS["train"]) // ACCUM)
warmup = CFG["WARMUP_EPOCHS"] * steps_per_epoch
total = CFG["EPOCHS"] * steps_per_epoch

def lr_at(step):
    if step < warmup:
        return step / max(warmup, 1)
    prog = (step - warmup) / max(total - warmup, 1)
    return max(0.01, 0.5 * (1 + math.cos(math.pi * prog)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_at)
scaler = torch.amp.GradScaler("cuda", enabled=(CFG["AMP_DTYPE"] == "fp16"))
ema = EMA(model, CFG["EMA_DECAY"]) if CFG["USE_EMA"] else None
print(f"  LR={LR:.2e} | {steps_per_epoch} opt-steps/epoch | warmup {warmup} | total {total}")

---
# 9 · Train / evaluate

In [ ]:
from tqdm.auto import tqdm

# Mid-epoch safety save. An epoch is ~8-12 min; without this a crash at minute 11
# throws that away. With it you lose at most CKPT_EVERY_MIN of work.
CKPT_EVERY_MIN = 10

def train_epoch(epoch, max_steps=None, ckpt_cb=None):
    global CKPT_EVERY_MIN
    model.train()
    tot, n, t0 = 0.0, 0, time.time()
    last_save = time.time()
    optimizer.zero_grad(set_to_none=True)
    nb = max_steps or len(LOADERS["train"])
    pbar = tqdm(LOADERS["train"], total=nb, desc=f"  epoch {epoch:02d}",
                unit="batch", dynamic_ncols=True, leave=True)
    for i, (x, y, w) in enumerate(pbar):
        if max_steps and i >= max_steps: break
        x = x.to(DEV, non_blocking=True)
        if CFG["CHANNELS_LAST"]: x = x.to(memory_format=torch.channels_last)
        y, w = y.to(DEV, non_blocking=True), w.to(DEV, non_blocking=True)
        with torch.autocast("cuda", dtype=AMP_DT):
            loss = criterion(model(x), y, w) / ACCUM
        if not torch.isfinite(loss):
            # save first so the run is diagnosable and resumable (audit fix J)
            if ckpt_cb:
                try: ckpt_cb()
                except Exception: pass
            raise RuntimeError(
                f"non-finite loss at epoch {epoch} step {i}. Lower BASE_LR or "
                f"POS_WEIGHT_CLAMP and resume.")
        scaler.scale(loss).backward() if scaler.is_enabled() else loss.backward()
        if (i + 1) % ACCUM == 0:
            if scaler.is_enabled(): scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), CFG["GRAD_CLIP"])
            if scaler.is_enabled(): scaler.step(optimizer); scaler.update()
            else: optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            if ema: ema.update(model)
        tot += loss.item() * ACCUM; n += 1
        pbar.set_postfix_str(
            f"loss={tot/max(n,1):.4f} lr={optimizer.param_groups[0]['lr']:.1e} "
            f"{(i+1)*BATCH/(time.time()-t0):.0f} img/s", refresh=False)

        # ---- mid-epoch safety save --------------------------------------
        if ckpt_cb and (time.time() - last_save) > CKPT_EVERY_MIN * 60:
            pbar.write(f"    [safety save @ step {i}/{nb}]")
            ckpt_cb()
            last_save = time.time()
    pbar.close()
    return tot / max(n, 1)

@torch.no_grad()
def predict(loader, net, max_batches=None):
    net.eval()
    P, Y = [], []
    for i, (x, y, _) in enumerate(loader):
        if max_batches and i >= max_batches: break
        x = x.to(DEV, non_blocking=True)
        if CFG["CHANNELS_LAST"]: x = x.to(memory_format=torch.channels_last)
        with torch.autocast("cuda", dtype=AMP_DT):
            P.append(torch.sigmoid(net(x)).float().cpu())
        Y.append(y)
    return torch.cat(P).numpy(), torch.cat(Y).numpy()

def auroc_per_class(y, p):
    out = {}
    for i, c in enumerate(PATHOLOGIES):
        out[c] = roc_auc_score(y[:, i], p[:, i]) if len(np.unique(y[:, i])) > 1 else float("nan")
    v = [x for x in out.values() if x == x]
    out["MEAN"] = float(np.mean(v)) if v else float("nan")
    return out

def best_thresholds(y, p):
    """F1-optimal threshold per class, fitted on VAL only (never on test)."""
    th = {}
    for i, c in enumerate(PATHOLOGIES):
        grid = np.arange(0.05, 0.96, 0.01)
        f1 = [f1_score(y[:, i], (p[:, i] >= t).astype(int), zero_division=0) for t in grid]
        th[c] = float(grid[int(np.argmax(f1))])
    return th

---
# 10 · Checkpointing (survives Colab disconnects)

In [ ]:
BEST = CKPT_DIR / "best.pt"          # Drive — survives VM loss
LAST = CKPT_DIR / "last.pt"          # Drive — written at each epoch end
# Mid-epoch safety saves go to LOCAL disk (audit fix G). A 1.4 GB Drive write
# takes 1-3 min; doing that every 10 min would burn ~13% of the run in I/O.
# /content survives a kernel restart or OOM (the common failures) and is lost
# only if the whole VM dies, which the epoch-end Drive save covers.
LOCAL_LAST = Path("/content/last_local.pt")

def save_ckpt(path, epoch, best_metric, history, ema_only=False):
    state = {"epoch": epoch, "best_metric": best_metric, "history": history,
             "cfg": CFG, "pathologies": PATHOLOGIES, "batch": BATCH, "accum": ACCUM,
             "model": model.state_dict(),
             "ema": ema.shadow if ema else None}
    if not ema_only:
        state |= {"optimizer": optimizer.state_dict(),
                  "scheduler": scheduler.state_dict(),
                  "scaler": scaler.state_dict()}
    tmp = path.with_suffix(".tmp")
    torch.save(state, tmp); tmp.replace(path)

def load_ckpt(path):
    c = torch.load(path, map_location="cpu", weights_only=False)
    model.load_state_dict(c["model"])
    if "optimizer" in c: optimizer.load_state_dict(c["optimizer"])
    if "scheduler" in c: scheduler.load_state_dict(c["scheduler"])
    if "scaler" in c: scaler.load_state_dict(c["scaler"])
    if ema and c.get("ema"): ema.shadow = {k: v.to(DEV) for k, v in c["ema"].items()}
    return c["epoch"], c["best_metric"], c.get("history", [])

start_epoch, best_metric, history = 0, -1.0, []
cands = [p for p in (LOCAL_LAST, LAST) if p.exists()]
if CFG["RESUME"] and cands and not CFG["SMOKE_TEST"]:
    src = max(cands, key=lambda p: p.stat().st_mtime)   # newest wins
    start_epoch, best_metric, history = load_ckpt(src)
    print(f"  ▶ resumed from {src.name}: epoch {start_epoch}, best mean AUROC {best_metric:.4f}")
else:
    print("  starting fresh")

---
# 11 · SMOKE TEST — exercise every path before spending hours

Runs ~40 steps and a partial eval. **Must print `SMOKE TEST PASSED`.**

In [ ]:
if CFG["SMOKE_TEST"]:
    print("=" * 80); print("  SMOKE TEST (~5 min, ≈0.4 CU)"); print("=" * 80)
    t0 = time.time()
    hits = []
    # force the mid-epoch save to fire during the smoke test, otherwise the
    # 10-minute timer never trips in 40 steps and the path goes untested (fix F)
    _saved_every = CKPT_EVERY_MIN
    CKPT_EVERY_MIN = 0
    l = train_epoch(0, max_steps=40, ckpt_cb=lambda: hits.append(1))
    CKPT_EVERY_MIN = _saved_every
    assert hits, "mid-epoch safety-save hook never fired"
    print(f"  progress bar + safety-save hook OK (fired {len(hits)}x)")
    print(f"  train loop OK — loss {l:.4f}")
    p, y = predict(LOADERS["val"], model, max_batches=10)
    a = auroc_per_class(y, p)
    print(f"  eval loop OK — {len(y)} samples, mean AUROC {a['MEAN']:.4f} (random ≈ 0.5)")
    th = best_thresholds(y, p); print(f"  threshold search OK — {list(th.items())[:2]} ...")
    save_ckpt(LAST, 0, a["MEAN"], []); print(f"  checkpoint save OK — {LAST.stat().st_size/1e6:.0f} MB")
    e0, b0, _ = load_ckpt(LAST); print(f"  checkpoint resume OK — epoch {e0}")
    if ema:
        # exercise the SAME swap path training uses — never deepcopy (fix B)
        w0 = model.classifier[-1].weight.detach().clone()
        ema.apply(model)
        assert not torch.equal(model.classifier[-1].weight, w0), "EMA apply did nothing"
        ema.restore(model)
        assert torch.equal(model.classifier[-1].weight, w0), "EMA restore failed"
        print("  EMA apply/restore OK (no second GPU model allocated)")
    ips = 40 * BATCH / (time.time() - t0)
    eta = len(DS["train"]) / max(ips, 1) * CFG["EPOCHS"] / 3600
    print(f"\n  throughput ≈ {ips:.0f} img/s")
    print(f"  ESTIMATED FULL RUN: {eta:.1f} h ≈ {eta*RATE:.0f} compute units")
    print("=" * 80)
    print("  ✅ SMOKE TEST PASSED")
    print("=" * 80)
    print("\n  NEXT: set SMOKE_TEST = False in cell 0, then Runtime → Restart and run all.")
    raise SystemExit("smoke test complete — flip SMOKE_TEST to False")

---
# 12 · Training

In [ ]:
print("=" * 80); print(f"  TRAINING  epochs {start_epoch+1} → {CFG['EPOCHS']}"); print("=" * 80)
patience = 0
for epoch in range(start_epoch + 1, CFG["EPOCHS"] + 1):
    te = time.time()
    # Mid-epoch saves record `epoch-1` as complete, so a resume replays this
    # epoch from its start with the current weights, optimizer and schedule.
    # Nothing is corrupted; at worst a few minutes of data ordering is repeated.
    tl = train_epoch(epoch, ckpt_cb=lambda: save_ckpt(LOCAL_LAST, epoch - 1, best_metric, history))
    # Evaluate on EMA weights without allocating a second GPU model (audit fix B)
    if ema: ema.apply(model)
    p, y = predict(LOADERS["val"], model)
    if ema: ema.restore(model)
    a = auroc_per_class(y, p)
    dt = time.time() - te
    elapsed = (time.time() - T_START) / 3600
    history.append({"epoch": epoch, "train_loss": tl, **a,
                    "lr": optimizer.param_groups[0]["lr"], "secs": dt})
    star = ""
    if a["MEAN"] > best_metric:
        best_metric = a["MEAN"]; patience = 0; star = "  ** BEST"
        save_ckpt(BEST, epoch, best_metric, history, ema_only=True)
    else:
        patience += 1
    save_ckpt(LAST, epoch, best_metric, history)
    print(f"\n  E{epoch:02d} loss={tl:.4f} | meanAUROC={a['MEAN']:.4f} "
          f"cardio={a['Cardiomegaly']:.4f} | {dt/60:.1f}min | "
          f"{elapsed:.1f}h ≈ {elapsed*RATE:.0f} CU{star}", flush=True)
    if epoch % 5 == 0 or star:
        print("    " + "  ".join(f"{c[:11]}={a[c]:.3f}" for c in PATHOLOGIES))
    if patience >= CFG["PATIENCE"]:
        print(f"\n  early stopping — no improvement for {CFG['PATIENCE']} epochs"); break

print(f"\n  DONE | best mean AUROC {best_metric:.4f} | "
      f"{(time.time()-T_START)/3600:.1f} h ≈ {(time.time()-T_START)/3600*RATE:.0f} CU")

---
# 13 · Test evaluation

Thresholds are fitted on **val** and applied to **test** — never fitted on test.

In [ ]:
if not BEST.exists():
    raise FileNotFoundError(
        f"{BEST} not found — no epoch completed with an improvement. "
        f"Check the training log above before evaluating.")
c = torch.load(BEST, map_location="cpu", weights_only=False)
model.load_state_dict(c["model"]); model.to(DEV)
if ema and c.get("ema"):
    ema.shadow = {k: v.to(DEV) for k, v in c["ema"].items()}; ema.copy_to(model)
print(f"  loaded best checkpoint (epoch {c['epoch']})")

pv, yv = predict(LOADERS["val"], model)
TH = best_thresholds(yv, pv)
pt, yt = predict(LOADERS["test"], model)
at = auroc_per_class(yt, pt)

print("\n" + "=" * 92); print("  TEST RESULTS"); print("=" * 92)
print(f"  {'pathology':<22}{'AUROC':>8}{'thr':>7}{'F1':>8}{'Prec':>8}{'Rec':>8}{'F1@0.5':>9}")
print("  " + "-" * 70)
rows = {}
for i, p_ in enumerate(PATHOLOGIES):
    t_ = TH[p_]; pred = (pt[:, i] >= t_).astype(int); pred5 = (pt[:, i] >= 0.5).astype(int)
    r = dict(AUROC=at[p_], threshold=t_,
             F1=f1_score(yt[:, i], pred, zero_division=0),
             Precision=precision_score(yt[:, i], pred, zero_division=0),
             Recall=recall_score(yt[:, i], pred, zero_division=0),
             F1_at_0_5=f1_score(yt[:, i], pred5, zero_division=0))
    rows[p_] = r
    print(f"  {p_:<22}{r['AUROC']:>8.4f}{t_:>7.2f}{r['F1']:>8.4f}"
          f"{r['Precision']:>8.4f}{r['Recall']:>8.4f}{r['F1_at_0_5']:>9.4f}")
print("  " + "-" * 70)
print(f"  {'MEAN':<22}{at['MEAN']:>8.4f}{'':>7}"
      f"{np.mean([r['F1'] for r in rows.values()]):>8.4f}")

# bootstrap CI on mean AUROC
rng = np.random.default_rng(0); boot = []
for _ in range(200):
    idx = rng.integers(0, len(yt), len(yt))
    try: boot.append(auroc_per_class(yt[idx], pt[idx])["MEAN"])
    except Exception: pass
print(f"\n  mean AUROC 95% CI: [{np.percentile(boot,2.5):.4f}, {np.percentile(boot,97.5):.4f}]")

PREV = {"AUROC (old run)": {"Cardiomegaly":.9235,"Edema":.8921,"Pleural_Effusion":.9153,
        "Atelectasis":.7800,"Consolidation":.7498,"Lung_Opacity":.7247,
        "Pneumonia":.7458,"Pneumothorax":.8696,"MEAN":.8251}}
print(f"\n  {'pathology':<22}{'OLD':>9}{'NEW':>9}{'Δ':>9}")
print("  " + "-" * 49)
for p_ in PATHOLOGIES + ["MEAN"]:
    o = PREV["AUROC (old run)"][p_]; n_ = at[p_]
    print(f"  {p_:<22}{o:>9.4f}{n_:>9.4f}{n_-o:>+9.4f}")

---
# 14 · Save artifacts

In [ ]:
res = {"stage": 5, "timestamp": datetime.now().isoformat(), "gpu": GPU,
       "best_epoch": int(c["epoch"]), "best_val_mean_auroc": float(best_metric),
       "test": {k: {kk: float(vv) for kk, vv in v.items()} for k, v in rows.items()},
       "test_mean_auroc": float(at["MEAN"]),
       "thresholds": TH, "config": CFG, "batch": BATCH, "accum": ACCUM,
       "hours": round((time.time()-T_START)/3600, 2),
       "compute_units_est": round((time.time()-T_START)/3600*RATE, 1),
       "history": history}
(REPORTS / "stage5_results.json").write_text(json.dumps(res, indent=2, default=str), encoding="utf-8")
(CKPT_DIR / "thresholds.json").write_text(json.dumps(TH, indent=2), encoding="utf-8")

# backbone-only export for Stage 4 (report generator vision encoder)
torch.save({"features": {k: v for k, v in model.state_dict().items() if k.startswith("features.")},
            "config": {"arch": CFG["ARCH"], "img_size": CFG["IMG_SIZE"],
                       "normalize": "per_image_zscore", "pathologies": PATHOLOGIES}},
           CKPT_DIR / "backbone_for_stage4.pt")

print(f"  ✅ {REPORTS/'stage5_results.json'}")
print(f"  ✅ {CKPT_DIR/'best.pt'}")
print(f"  ✅ {CKPT_DIR/'thresholds.json'}")
print(f"  ✅ {CKPT_DIR/'backbone_for_stage4.pt'}  ← Stage 4 loads this")
print(f"\n  total {(time.time()-T_START)/3600:.2f} h ≈ {(time.time()-T_START)/3600*RATE:.0f} compute units")
print("\n  ⚠️  Runtime → Disconnect and delete runtime  — units burn while connected.")